# 🔬 Notebook 03: Leakage Audit & Rigorous Data Splitting Protocols
**AI-Enabled Nutrient Balancing System for Hydroponics**  
*Sprint 3: Methodology and Evaluation Protocol Hardening*

---

## 1. Executive Summary & Problem Formulation

In time-series forecasting for IoT agricultural control systems, conventional machine learning evaluation protocols often produce catastrophic methodological vulnerabilities:
1. **Scaling Leakage:** Fitting scalers (e.g. `MinMaxScaler`) on the full dataset before splitting leaks distributional parameters (minima and maxima) of test sets into the model's training space.
2. **Evaluation Leakage:** Supplying the test holdout as `validation_data` during training permits validation loss tracking and early-stopping hyperparameter tuning against the test set.
3. **Autocorrelation & Temporal Leakage:** Shuffling sequences randomly across partitions creates extreme overlap (e.g., sequences $t$ and $t+1$ share $W-1$ out of $W$ timesteps). With sensor lag-1 autocorrelation exceeding 0.99, the model merely performs local interpolation rather than true out-of-sample forecasting.
4. **Discontinuity Slicing:** Naive sliding-window sequence generators slice blindly across multi-hour system shutdowns and manual nutrient dosing interventions, generating physically corrupt transitions.

In this notebook, we systematically evaluate three split protocols:
- **Strategy 1: Random Split (Negative Control)** — Demonstrates the illusion of high accuracy caused by temporal autocorrelation leakage.
- **Strategy 2: Chronological Split (70/10/20)** — Strict temporal ordering, train-only scaler fitting, boundary-aware sequences, and true validation holdout.
- **Strategy 3: Group/Session Split** — Physical run holdout testing cross-session generalization across operational regimes.

In [ ]:
import os
import sys
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Ensure project root in sys.path
ROOT_DIR = os.path.dirname(os.path.abspath(''))
if ROOT_DIR not in sys.path:
    sys.path.insert(0, ROOT_DIR)

from src.config import RAW_CSV, FEATURES, TARGET_COLS, TIME_STEPS, TOLERANCES
from src.preprocessing import build_pipeline, detect_session_segments
from src.models.baseline_cnn_bilstm import build_cnn_bilstm
from src.evaluation import compute_regression_metrics, evaluate_tds_sensitivity, format_comparison_table

print("All modules loaded successfully.")

## 2. Load Dataset & Audit Temporal Discontinuities
We load the un-interpolated dataset and inspect the operational sessions separated by system shutdowns.

In [ ]:
df = pd.read_csv(RAW_CSV)
print(f"Raw dataset shape: {df.shape}")

# Detect major operational sessions (> 1 hour gaps)
sessions = detect_session_segments(df, max_gap_seconds=3600.0)
print(f"Detected {len(sessions)} operational sessions:")
for i, (s, e) in enumerate(sessions):
    dt_s = df.loc[s, 'timestamp']
    dt_e = df.loc[e-1, 'timestamp']
    print(f"  Session {i+1}: Rows {s:5d} to {e-1:5d} ({e-s:5d} rows) | {dt_s} -> {dt_e}")

## 3. Preprocessing Pipeline Across 3 Split Protocols
We instantiate all three protocols using our hardened `build_pipeline` module.

In [ ]:
pipe_rand = build_pipeline(df, split_strategy='random', verbose=True)
pipe_chro = build_pipeline(df, split_strategy='chronological', verbose=True)
pipe_grp  = build_pipeline(df, split_strategy='group', verbose=True)

## 4. Empirical Evaluation Results
Load the experimental results logged by Sprint 3 comparison pipeline (`experiments/exp_003_leakage_comparison.json`).

In [ ]:
exp_file = os.path.join(ROOT_DIR, 'experiments', 'exp_003_leakage_comparison.json')
if os.path.exists(exp_file):
    with open(exp_file, 'r') as f:
        exp_data = json.load(f)
    df_comp = pd.DataFrame(exp_data['comparison_table'])
    display(df_comp)
else:
    print("Run notebooks/03_leakage_comparison.py first to populate experiment records.")

## 5. Comparative Visualizations
Display the comparison charts generated for the research paper.

In [ ]:
from IPython.display import Image, display

fig_dir = os.path.join(ROOT_DIR, 'reports', 'figures')
figures = [
    ('MAE Comparison Across Protocols', 'fig09_split_strategy_mae_comparison.png'),
    ('R² Comparison by Sensor', 'fig10_split_strategy_r2_comparison.png'),
    ('Convergence & Generalization Curves', 'fig11_strategy_training_histories.png'),
    ('Residual Distributions (Leaked vs Rigorous)', 'fig12_leakage_residual_distribution.png')
]

for title, fname in figures:
    fpath = os.path.join(fig_dir, fname)
    if os.path.exists(fpath):
        print(f"### {title}")
        display(Image(filename=fpath))
    else:
        print(f"Figure {fname} not found yet.")

## 6. Key Scientific Conclusions for Peer Review

1. **The Autocorrelation Trap:**  
   Random splitting in IoT time series achieves an artificial $R^2 > 0.99$ across all sensors because adjacent sequences share 14 out of 15 timesteps. This does not represent model generalization; it represents sequence leakage.
2. **Rigorous Chronological Protocol:**  
   A chronological 70/10/20 partition with train-only scaling and boundary-aware sequencing produces defensible, realistic metrics that accurately reflect online sensor forecasting.
3. **Domain Shift in Operational Sessions:**  
   Evaluating on held-out physical sessions demonstrates how external interventions (such as manual dosing and reservoir level changes) create genuine domain shift, requiring online adaptation or uncertainty quantification.